### Clean driver license data (+ AddressStreet)

In [1]:
import argparse
import re
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import bigquery

# ------------- helpers -------------
def to_str(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("")

def rm_accents(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).map(
        lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("ASCII")
    ).astype("string")

def collapse_ws(s: pd.Series) -> pd.Series:
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

# keep-lists
PUNCT_NAME_DROP = r"[^A-Z'\-\s]"      # drop anything not A-Z, space, apostrophe, hyphen
PUNCT_CITY_DROP = r"[^A-Z'\-\s]"

STATE_MAP = {
    "ALABAMA":"AL","ALASKA":"AK","ARIZONA":"AZ","ARKANSAS":"AR","CALIFORNIA":"CA","COLORADO":"CO",
    "CONNECTICUT":"CT","DELAWARE":"DE","FLORIDA":"FL","GEORGIA":"GA","HAWAII":"HI","IDAHO":"ID",
    "ILLINOIS":"IL","INDIANA":"IN","IOWA":"IA","KANSAS":"KS","KENTUCKY":"KY","LOUISIANA":"LA",
    "MAINE":"ME","MARYLAND":"MD","MASSACHUSETTS":"MA","MICHIGAN":"MI","MINNESOTA":"MN","MISSISSIPPI":"MS",
    "MISSOURI":"MO","MONTANA":"MT","NEBRASKA":"NE","NEVADA":"NV","NEW HAMPSHIRE":"NH","NEW JERSEY":"NJ",
    "NEW MEXICO":"NM","NEW YORK":"NY","NORTH CAROLINA":"NC","NORTH DAKOTA":"ND","OHIO":"OH","OKLAHOMA":"OK",
    "OREGON":"OR","PENNSYLVANIA":"PA","RHODE ISLAND":"RI","SOUTH CAROLINA":"SC","SOUTH DAKOTA":"SD",
    "TENNESSEE":"TN","TEXAS":"TX","UTAH":"UT","VERMONT":"VT","VIRGINIA":"VA","WASHINGTON":"WA",
    "WEST VIRGINIA":"WV","WISCONSIN":"WI","WYOMING":"WY","DISTRICT OF COLUMBIA":"DC","WASHINGTON DC":"DC","DC":"DC"
}
DIR_MAP = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
STREET_MAP = {
    "STREET":"ST","AVENUE":"AVE","ROAD":"RD","LANE":"LN","DRIVE":"DR","BOULEVARD":"BLVD","COURT":"CT",
    "TERRACE":"TER","PLACE":"PL","SQUARE":"SQ","PARKWAY":"PKWY","CIRCLE":"CIR","TRAIL":"TRL",
    "HIGHWAY":"HWY","EXPRESSWAY":"EXPY","FREEWAY":"FWY","CENTER":"CTR","CENTRE":"CTR","PLAZA":"PLZ",
    "RIDGE":"RDG","MOUNT":"MT","MOUNTAIN":"MTN","ALLEY":"ALY","WAY":"WAY"
}
TITLE_RE = r"\b(MR|MRS|MS|MISS|DR|PROF|REV|SRGT|SGT|OFFICER|ATTY|ATTORNEY|JUDGE)\b"

def clean_names(df: pd.DataFrame) -> pd.Series:
    for c in ["FirstName","MiddleName","LastName","SuffixName"]:
        df[c] = to_str(df[c])
    # accents → ASCII
    for c in ["FirstName","MiddleName","LastName","SuffixName"]:
        df[c] = rm_accents(df[c]).str.upper()
        if c != "SuffixName":
            df[c] = df[c].str.replace(TITLE_RE, "", regex=True)
        df[c] = collapse_ws(df[c].str.replace(PUNCT_NAME_DROP, "", regex=True))

    suffix_map = {"JR":"JR","JUNIOR":"JR","SR":"SR","SENIOR":"SR","II":"II","2ND":"II","III":"III","3RD":"III","IV":"IV","4TH":"IV","V":"V","5TH":"V","VI":"VI","6TH":"VI"}
    df["SuffixName"] = df["SuffixName"].map(suffix_map).fillna("")

    full = (
        df["FirstName"].str.cat(df["MiddleName"], sep=" ", na_rep="")
                       .str.cat(df["LastName"],  sep=" ", na_rep="")
                       .str.cat(df["SuffixName"], sep=" ", na_rep="")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip()
    )
    return full

def clean_address(df: pd.DataFrame) -> pd.DataFrame:
    # street lines (1–3) → AddressStreet
    for c in ["Res_Address1","Res_Address2","Res_Address3","Res_City","Res_State","Res_ZipCodeFull","Res_ZipCode5"]:
        if c in df.columns:
            df[c] = to_str(df[c])

    # combine street parts then normalize
    street = (df.get("Res_Address1", "") + " " + df.get("Res_Address2", "") + " " + df.get("Res_Address3", ""))
    street = rm_accents(street).str.upper()
    # PO BOX, units, hash→APT, punctuation
    street = street.str.replace(r"\bP\.?\s*O\.?\s*B(?:OX)?\b", "PO BOX", regex=True)
    street = street.str.replace(r"\b(APARTMENT|APT)\b", "APT", regex=True)\
                   .str.replace(r"\b(SUITE|STE)\b", "STE", regex=True)\
                   .str.replace(r"#\s*([A-Z0-9\-]+)", r"APT \1", regex=True)\
                   .str.replace(r"[.,]", "", regex=True)
    # directionals + street types
    street = street.str.replace(
        r"\b(NORTHWEST|NORTHEAST|SOUTHWEST|SOUTHEAST|NORTH|SOUTH|EAST|WEST)\b",
        lambda m: DIR_MAP[m.group(0)], regex=True
    )
    st_pat = r"\b(" + "|".join(map(re.escape, STREET_MAP.keys())) + r")\b"
    street = street.str.replace(st_pat, lambda m: STREET_MAP[m.group(0)], regex=True)
    street = collapse_ws(street)

    city  = rm_accents(df.get("Res_City", "")).str.upper()
    city  = collapse_ws(city.str.replace(PUNCT_CITY_DROP, "", regex=True))

    state = df.get("Res_State", "").str.upper()
    state = state.map(STATE_MAP).fillna(state)
    state = state.str.replace(r"[^A-Z]", "", regex=True).str[:2]

    zip5  = df.get("Res_ZipCode5", "")
    zfull = df.get("Res_ZipCodeFull", "")
    zip5  = zip5.where(zip5.str.len() > 0, zfull)
    zip5  = zip5.str.replace(r"[^0-9]", "", regex=True).str[:5]

    fulladdr = (
        street.str.cat(city,  sep=", ", na_rep="")
              .str.cat(state, sep=", ", na_rep="")
              .str.cat(zip5,  sep=", ", na_rep="")
              .str.replace(r"(,\s*){2,}", ", ", regex=True)
              .str.strip(", ").str.strip()
    )

    return pd.DataFrame({
        "AddressStreet": street,
        "Res_City": city,
        "Res_State": state,
        "Zip5": zip5,
        "FullAddress": fulladdr
    })

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--bq_table", default="omes-datascience-sbx.Unclaimed_Property_Analysis.DLMatch0616")
    parser.add_argument("--out_parquet", default="driver_dl_for_ann.parquet")
    args, _ = parser.parse_known_args()

    # Read raw columns (fast; cleaning is all in Python)
    query = f"""
    SELECT
      CAST(DLN AS STRING) AS DLN,
      FirstName, MiddleName, LastName, SuffixName,
      Res_Address1, Res_Address2, Res_Address3,
      Res_City, Res_State, Res_ZipCodeFull, Res_ZipCode5
    FROM `{args.bq_table}`
    """
    client = bigquery.Client()
    try:
        from google.cloud import bigquery_storage  # noqa: F401
        df = client.query(query).to_dataframe(create_bqstorage_client=True)
    except Exception:
        df = client.query(query).to_dataframe()

    # Ensure string DLN
    df["DLN"] = to_str(df["DLN"])

    # Clean
    full_name = clean_names(df)
    addr_df   = clean_address(df)

    out = pd.DataFrame({
        "DLN": df["DLN"].astype(str),
        "FullName": full_name,
    }).join(addr_df[["AddressStreet","Res_City","Res_State","Zip5","FullAddress"]])

    # Write (compressed) and print only the count
    pq.write_table(pa.Table.from_pandas(out, preserve_index=False), args.out_parquet, compression="zstd")
    print(len(out))

if __name__ == "__main__":
    main()


5972828


### Clean unclaimed data (+ AddressStreet)

In [2]:
import argparse
import re
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import bigquery

# ------------- helpers -------------
def to_str(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("")

def rm_accents(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).map(
        lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("ASCII")
    ).astype("string")

def collapse_ws(s: pd.Series) -> pd.Series:
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

# keep-lists
PUNCT_NAME_DROP = r"[^A-Z'\-\s]"      # drop anything not A-Z, space, apostrophe, hyphen
PUNCT_CITY_DROP = r"[^A-Z'\-\s]"

STATE_MAP = {
    "ALABAMA":"AL","ALASKA":"AK","ARIZONA":"AZ","ARKANSAS":"AR","CALIFORNIA":"CA","COLORADO":"CO",
    "CONNECTICUT":"CT","DELAWARE":"DE","FLORIDA":"FL","GEORGIA":"GA","HAWAII":"HI","IDAHO":"ID",
    "ILLINOIS":"IL","INDIANA":"IN","IOWA":"IA","KANSAS":"KS","KENTUCKY":"KY","LOUISIANA":"LA",
    "MAINE":"ME","MARYLAND":"MD","MASSACHUSETTS":"MA","MICHIGAN":"MI","MINNESOTA":"MN","MISSISSIPPI":"MS",
    "MISSOURI":"MO","MONTANA":"MT","NEBRASKA":"NE","NEVADA":"NV","NEW HAMPSHIRE":"NH","NEW JERSEY":"NJ",
    "NEW MEXICO":"NM","NEW YORK":"NY","NORTH CAROLINA":"NC","NORTH DAKOTA":"ND","OHIO":"OH","OKLAHOMA":"OK",
    "OREGON":"OR","PENNSYLVANIA":"PA","RHODE ISLAND":"RI","SOUTH CAROLINA":"SC","SOUTH DAKOTA":"SD",
    "TENNESSEE":"TN","TEXAS":"TX","UTAH":"UT","VERMONT":"VT","VIRGINIA":"VA","WASHINGTON":"WA",
    "WEST VIRGINIA":"WV","WISCONSIN":"WI","WYOMING":"WY","DISTRICT OF COLUMBIA":"DC","WASHINGTON DC":"DC","DC":"DC"
}
DIR_MAP = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
STREET_MAP = {
    "STREET":"ST","AVENUE":"AVE","ROAD":"RD","LANE":"LN","DRIVE":"DR","BOULEVARD":"BLVD","COURT":"CT",
    "TERRACE":"TER","PLACE":"PL","SQUARE":"SQ","PARKWAY":"PKWY","CIRCLE":"CIR","TRAIL":"TRL",
    "HIGHWAY":"HWY","EXPRESSWAY":"EXPY","FREEWAY":"FWY","CENTER":"CTR","CENTRE":"CTR","PLAZA":"PLZ",
    "RIDGE":"RDG","MOUNT":"MT","MOUNTAIN":"MTN","ALLEY":"ALY","WAY":"WAY"
}
TITLE_RE = r"\b(MR|MRS|MS|MISS|DR|PROF|REV|SRGT|SGT|OFFICER|ATTY|ATTORNEY|JUDGE)\b"

def clean_names_unclaimed(df: pd.DataFrame) -> pd.Series:
    # Ensure strings
    for c in ["NameFirst","NameMiddle","NameLast","NameSuffix","FullName"]:
        df[c] = to_str(df[c])

    # Accents → ASCII, uppercase
    for c in ["NameFirst","NameMiddle","NameLast","NameSuffix","FullName"]:
        df[c] = rm_accents(df[c]).str.upper()

    # Remove titles & non-allowed chars from parts (not from FullName fallback)
    for c in ["NameFirst","NameMiddle","NameLast"]:
        df[c] = df[c].str.replace(TITLE_RE, "", regex=True)
        df[c] = collapse_ws(df[c].str.replace(PUNCT_NAME_DROP, "", regex=True))
    df["NameSuffix"] = collapse_ws(df["NameSuffix"].str.replace(PUNCT_NAME_DROP, "", regex=True))

    # Normalize suffixes
    suffix_map = {"JR":"JR","JUNIOR":"JR","SR":"SR","SENIOR":"SR","II":"II","2ND":"II","III":"III","3RD":"III","IV":"IV","4TH":"IV","V":"V","5TH":"V","VI":"VI","6TH":"VI"}
    df["NameSuffix"] = df["NameSuffix"].map(suffix_map).fillna(df["NameSuffix"])

    # Build from parts
    parts = (
        df["NameFirst"].str.cat(df["NameMiddle"], sep=" ", na_rep="")
                        .str.cat(df["NameLast"],  sep=" ", na_rep="")
                        .str.cat(df["NameSuffix"], sep=" ", na_rep="")
                        .str.replace(r"\s+", " ", regex=True)
                        .str.strip()
    )

    # Fallback to provided FullName if parts empty
    full_clean = collapse_ws(df["FullName"].str.replace(PUNCT_NAME_DROP, "", regex=True))
    out = parts.where(parts.str.len() > 0, full_clean)
    return out.rename("FullName_cleaned")

def clean_address_unclaimed(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["Address1","City","State","ZipCode"]:
        df[c] = to_str(df[c])

    # Street
    street = rm_accents(df["Address1"]).str.upper()
    street = street.str.replace(r"\bP\.?\s*O\.?\s*B(?:OX)?\b", "PO BOX", regex=True)
    street = street.str.replace(r"\b(APARTMENT|APT)\b", "APT", regex=True)\
                   .str.replace(r"\b(SUITE|STE)\b", "STE", regex=True)\
                   .str.replace(r"#\s*([A-Z0-9\-]+)", r"APT \1", regex=True)\
                   .str.replace(r"[.,]", "", regex=True)
    # Directionals + street types
    street = street.str.replace(
        r"\b(NORTHWEST|NORTHEAST|SOUTHWEST|SOUTHEAST|NORTH|SOUTH|EAST|WEST)\b",
        lambda m: DIR_MAP[m.group(0)], regex=True
    )
    st_pat = r"\b(" + "|".join(map(re.escape, STREET_MAP.keys())) + r")\b"
    street = street.str.replace(st_pat, lambda m: STREET_MAP[m.group(0)], regex=True)
    street = collapse_ws(street)

    # City / State / Zip
    city  = rm_accents(df["City"]).str.upper()
    city  = collapse_ws(city.str.replace(PUNCT_CITY_DROP, "", regex=True))

    state = df["State"].str.upper()
    state = state.map(STATE_MAP).fillna(state)
    state = state.str.replace(r"[^A-Z]", "", regex=True).str[:2]

    zip5  = df["ZipCode"].str.replace(r"[^0-9]", "", regex=True).str[:5]

    fulladdr = (
        street.str.cat(city,  sep=", ", na_rep="")
              .str.cat(state, sep=", ", na_rep="")
              .str.cat(zip5,  sep=", ", na_rep="")
              .str.replace(r"(,\s*){2,}", ", ", regex=True)
              .str.strip(", ").str.strip()
    )

    return pd.DataFrame({
        "AddressStreet": street.rename("AddressStreet"),
        "City": city.rename("City"),
        "State": state.rename("State"),
        "Zip5": zip5.rename("Zip5"),
        "FullAddress_cleaned": fulladdr.rename("FullAddress_cleaned")
    })

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--bq_table", default="omes-datascience-sbx.Unclaimed_Property_Analysis.unclaimed_property_new_data_20250730")
    parser.add_argument("--out_parquet", default="unclaimed_for_ann.parquet")
    args, _ = parser.parse_known_args()  # Jupyter-safe

    # Read required columns
    query = f"""
    SELECT
      HolderName, PropertyID, OwnerID, PropertyTypeCD, PropertyTypeCodeDescription,
      FullName, NameFirst, NameMiddle, NameLast, NameSuffix,
      Address1, City, State, ZipCode, Value, OwnerRelationshipTypeCodeDescription
    FROM `{args.bq_table}`
    """
    client = bigquery.Client()
    try:
        from google.cloud import bigquery_storage  # noqa: F401
        df = client.query(query).to_dataframe(create_bqstorage_client=True)
    except Exception:
        df = client.query(query).to_dataframe()

    # Drop rows missing OwnerID (cannot form UnclaimedUniqueID)
    if "OwnerID" in df.columns:
        df = df[~df["OwnerID"].isna()].copy()

    # Build UnclaimedUniqueID = PropertyID_OwnerID
    df["PropertyID"] = to_str(df["PropertyID"])
    df["OwnerID"]    = to_str(df["OwnerID"])
    df["UnclaimedUniqueID"] = (df["PropertyID"] + "_" + df["OwnerID"]).str.strip("_")

    # Clean names and address
    full_name = clean_names_unclaimed(df)
    addr_df   = clean_address_unclaimed(df)
    
    out = pd.concat(
    [
        df[["UnclaimedUniqueID",
            "PropertyTypeCodeDescription",
            "Value"]].reset_index(drop=True),
        full_name.reset_index(drop=True),
        addr_df.reset_index(drop=True),
    ],
    axis=1
)


    # Write (compressed) and print only the count
    pq.write_table(pa.Table.from_pandas(out, preserve_index=False), args.out_parquet, compression="zstd")
    print(len(out))

if __name__ == "__main__":
    main()


1045528


### Name embeddings + FAISS matching

In [3]:
#!/usr/bin/env python3
# step3_name_ann.py — Name matching with embeddings + FAISS IVF-PQ (CPU/GPU robust)
# Specs:
#   • Model: multi-qa-MiniLM-L6-cos-v1 (fallback: all-MiniLM-L6-v2)
#   • Index: OPQ16, IVF8192, PQ16x8; cosine via normalized embeddings, nprobe=64
#   • Keep matches with cosine ≥ 0.90
# Inputs:
#   • driver_dl_for_ann.parquet           (DLN, FullName)
#   • unclaimed_for_ann.parquet           (UnclaimedUniqueID, FullName_cleaned)
# Output:
#   • name_ann_matches.parquet            (DLN, UnclaimedUniqueID, NameScore)

import argparse
import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer

PLACEHOLDER_PAT = r"^(|N/?A|NONE|UNKNOWN|UNK|NO NAME|\.)$"

def load_model(primary="multi-qa-MiniLM-L6-cos-v1", fallback="all-MiniLM-L6-v2"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        return SentenceTransformer(primary, device=device)
    except Exception:
        return SentenceTransformer(fallback, device=device)

def embed_texts(model: SentenceTransformer, texts, batch=1024) -> np.ndarray:
    vecs = model.encode(
        texts,
        batch_size=batch,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalized → IP == cosine
    )
    return vecs.astype("float32")

def build_index(dim: int):
    """Create OPQ16,IVF8192,PQ16x8 index; move to GPU only if GPU symbols exist."""
    index = faiss.index_factory(dim, "OPQ16,IVF8192,PQ16x8", faiss.METRIC_INNER_PRODUCT)
    gpu_ok = (
        torch.cuda.is_available()
        and hasattr(faiss, "StandardGpuResources")
        and hasattr(faiss, "index_cpu_to_gpu")
    )
    if gpu_ok:
        try:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            return index, True
        except Exception:
            pass  # fall through to CPU
    return index, False

def set_nprobe(index, nprobe: int):
    """Set nprobe for both GPU and CPU (IndexIVF or wrapped in IndexPreTransform)."""
    # GPU index
    if hasattr(index, "setNumProbes"):
        try:
            index.setNumProbes(nprobe)
            return
        except Exception:
            pass
    # CPU IVF or wrapped
    try:
        ivf = faiss.extract_index_ivf(index)
        if ivf is not None:
            ivf.nprobe = nprobe
            return
    except Exception:
        pass
    # Last resort: if it's a plain IndexIVF
    if isinstance(index, faiss.IndexIVF):
        index.nprobe = nprobe

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--driver_parquet", default="driver_dl_for_ann.parquet")
    ap.add_argument("--unclaimed_parquet", default="unclaimed_for_ann.parquet")
    ap.add_argument("--out_parquet", default="name_ann_matches.parquet")
    ap.add_argument("--k", type=int, default=100)              # top-K per driver name
    ap.add_argument("--thresh", type=float, default=0.90)      # cosine threshold
    ap.add_argument("--embed_batch", type=int, default=1024)   # embedding batch size
    ap.add_argument("--train_sample", type=int, default=300000) # max train vectors
    ap.add_argument("--nprobe", type=int, default=64)
    args, _ = ap.parse_known_args()  # Jupyter-safe

    # -------- Load data
    drv = pd.read_parquet(args.driver_parquet, columns=["DLN","FullName"]).dropna(subset=["FullName"])
    uc  = pd.read_parquet(args.unclaimed_parquet, columns=["UnclaimedUniqueID","FullName_cleaned"]).dropna(subset=["FullName_cleaned"])

    # Filter placeholders
    drv = drv[~drv["FullName"].str.upper().str.strip().str.match(PLACEHOLDER_PAT, na=False)]
    uc  = uc[~uc["FullName_cleaned"].str.upper().str.strip().str.match(PLACEHOLDER_PAT, na=False)]

    # Ensure strings
    drv["DLN"] = drv["DLN"].astype(str)
    drv["FullName"] = drv["FullName"].astype(str)
    uc["UnclaimedUniqueID"] = uc["UnclaimedUniqueID"].astype(str)
    uc["FullName_cleaned"] = uc["FullName_cleaned"].astype(str)

    if len(drv) == 0 or len(uc) == 0:
        pd.DataFrame(columns=["DLN","UnclaimedUniqueID","NameScore"]).to_parquet(args.out_parquet, index=False)
        print(f"queries={len(drv)}, corpus={len(uc)}, kept=0 -> {args.out_parquet}")
        return

    # -------- Embeddings
    model = load_model()
    corpus_texts = uc["FullName_cleaned"].tolist()
    query_texts  = drv["FullName"].tolist()

    emb_c = embed_texts(model, corpus_texts, batch=args.embed_batch)  # (Nc, d)
    emb_q = embed_texts(model, query_texts,  batch=args.embed_batch)  # (Nq, d)
    dim   = emb_c.shape[1]
    Nc    = emb_c.shape[0]

    # -------- FAISS index
    faiss.omp_set_num_threads(8)
    index, on_gpu = build_index(dim)

    # Train (IVF/PQ requires training)
    if args.train_sample and Nc > args.train_sample:
        rng = np.random.default_rng(42)
        sel = rng.choice(Nc, size=args.train_sample, replace=False)
        train_vecs = emb_c[sel]
    else:
        train_vecs = emb_c

    try:
        index.train(train_vecs)
        index.add(emb_c)
        set_nprobe(index, args.nprobe)
    except Exception:
        # Fallback to a flat IP index if IVF-PQ training fails for any reason
        index = faiss.IndexFlatIP(dim)
        index.add(emb_c)

    # -------- Search
    topk = min(max(1, args.k), Nc)
    D, I = index.search(emb_q, topk)  # D: cosine sims, I: corpus idx

    # -------- Collect >= threshold
    dlns = drv["DLN"].to_numpy()
    uids = uc["UnclaimedUniqueID"].to_numpy()

    keep_rows = []
    for qi in range(D.shape[0]):
        sims = D[qi]
        idxs = I[qi]
        mask = (idxs != -1) & (sims >= args.thresh)
        if np.any(mask):
            for sc, ci in zip(sims[mask], idxs[mask]):
                keep_rows.append((dlns[qi], uids[ci], float(sc)))

    out = pd.DataFrame(keep_rows, columns=["DLN","UnclaimedUniqueID","NameScore"]).drop_duplicates()
    out.to_parquet(args.out_parquet, index=False)

    print(f"queries={len(drv)}, corpus={len(uc)}, kept={len(out)} -> {args.out_parquet}")

if __name__ == "__main__":
    main()


WARNING clustering 300000 points to 8192 centroids: please provide at least 319488 training points


queries=5967106, corpus=1023215, kept=854274 -> name_ann_matches.parquet


### Address embeddings + FAISS matching

In [4]:
#!/usr/bin/env python3
# ==========================================================
# step4_address_ann_fast.py — Address matching with blocking + embeddings
#
# Inputs (defaults now match your cleaners):
#   - driver_dl_for_ann.parquet        : DLN, AddressStreet, AddressCity|Res_City, AddressState|Res_State, AddressZip|Zip5
#   - unclaimed_for_ann.parquet (uses UnclaimedUniqueID, FullAddress_cleaned)
#   - address_ann_matches.parquet : DLN, UnclaimedUniqueID, AddressScore
#
# Key speedups:
#   1) Vectorized Tier-1 exact join on (ZIP|CITY|STATE, AddressStreet)
#   2) GLOBAL de-dup of street strings → embed once, reuse everywhere
#   3) GPU FAISS (FlatIP) for large blocks, NumPy matmul for small
#   4) House-number sub-blocking to shrink comparisons per block
#   5) Early short-circuit: stop after a driver gets any match in a tier
# ==========================================================

import os, sys, argparse, gc
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer

# ---------------- Config knobs (ENV or CLI) ----------------
# Default knobs for Address ANN matching (override via ENV or CLI)

DEF_THRESH = float(os.getenv("ADDR_THRESH", "0.95"))  
DEF_K = int(os.getenv("ADDR_K", "100"))  # candidates per query within a block
DEF_EMB_B = int(os.getenv("ADDR_EMBED_BATCH", "4096" if (torch.cuda.is_available() and faiss.get_num_gpus()>0) else "1024"))
DEF_SMALL = int(os.getenv("ADDR_SMALL_BLOCK_CUTOFF", "4000"))  # <= matmul; > FAISS
DEF_STOP = bool(int(os.getenv("ADDR_STOP_AT_FIRST_TIER", "1")))  # stop after a driver matches in a tier
DEF_USE_HN = bool(int(os.getenv("ADDR_USE_HOUSENUM", "1")))      # sub-block by leading house number

PRIMARY_MODEL  = os.getenv("ADDR_MODEL", "all-MiniLM-L6-v2")
FALLBACK_MODEL = os.getenv("ADDR_MODEL_FALLBACK", "multi-qa-MiniLM-L6-cos-v1")
PLACEHOLDER_ADDR = r"^(|N/?A|NONE|UNKNOWN|UNK|NO ADDRESS|ADDRESS UNKNOWN|\.)$"

# ---------------- Utilities ----------------
def parse_args():
    ap = argparse.ArgumentParser("Address ANN with blocking (fast, GPU-aware)")
    # UPDATED DEFAULTS ↓↓↓
    ap.add_argument("--driver_parquet",    default=os.getenv("DRIVER_ADDR_PARQUET", "driver_dl_for_ann.parquet"))
    ap.add_argument("--unclaimed_parquet", default=os.getenv("UNCLAIMED_ADDR_PARQUET", "unclaimed_for_ann.parquet"))
    ap.add_argument("--out_parquet",       default=os.getenv("ADDR_OUT_PARQUET", "address_ann_matches.parquet"))
    ap.add_argument("--thresh", type=float, default=DEF_THRESH)
    ap.add_argument("--k", type=int, default=DEF_K)
    ap.add_argument("--embed_batch", type=int, default=DEF_EMB_B)
    ap.add_argument("--small_block_cutoff", type=int, default=DEF_SMALL)
    ap.add_argument("--stop_at_first_tier_match", action="store_true" if DEF_STOP else "store_false")
    ap.add_argument("--use_house_number", action="store_true" if DEF_USE_HN else "store_false")
    ap.add_argument("--seed", type=int, default=int(os.getenv("SEED","123")))
    args, _ = ap.parse_known_args()
    return args

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Missing required column; tried {candidates}")
    return None

def norm_key(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

def safe_zip5(s: pd.Series) -> pd.Series:
    z = s.fillna("").astype(str).str.replace(r"[^0-9]", "", regex=True).str[:5]
    return z.where(z != "00000", "")

def house_num(s: pd.Series) -> pd.Series:
    num = s.str.extract(r"\b(\d+)\b", expand=True)[0]
    return num.fillna("")

def load_model(primary=PRIMARY_MODEL, fallback=FALLBACK_MODEL):
    device = "cuda" if (torch.cuda.is_available() and faiss.get_num_gpus()>0) else "cpu"
    try:
        return SentenceTransformer(primary, device=device)
    except Exception:
        return SentenceTransformer(fallback, device=device)

def embed_norm(model: SentenceTransformer, texts, batch=1024) -> np.ndarray:
    vecs = model.encode(
        texts,
        batch_size=batch,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2-norm → IP == cosine
    )
    return vecs.astype("float32")

# ---------------- Global embedding (dedup once) ----------------
def dedup_and_embed(model, ser_text: pd.Series, batch: int):
    codes, uniques = pd.factorize(ser_text, sort=False)
    emb = embed_norm(model, uniques.tolist(), batch=batch)
    return codes.astype(np.int32), uniques, emb

# ---------------- Fast exact match (Tier-1) ----------------
def exact_join_tier1(drv, uc, cols):
    key = "key1"
    drv[key] = drv[cols.zipc].str.cat(drv[cols.city], sep="|").str.cat(drv[cols.state], sep="|")
    uc[key]  = uc[cols.zipc].str.cat(uc[cols.city],  sep="|").str.cat(uc[cols.state],  sep="|")
    left  = drv[[cols.dln, "AddressStreet", key]].drop_duplicates()
    right = uc [[ "UnclaimedUniqueID","AddressStreet", key]].drop_duplicates()
    m = left.merge(right, on=[key,"AddressStreet"], how="inner")
    if not m.empty:
        m = m[[cols.dln, "UnclaimedUniqueID"]].drop_duplicates()
        m["AddressScore"] = 1.0
    drv.drop(columns=[key], inplace=True)
    uc.drop(columns=[key], inplace=True)
    return m

# ---------------- Block builders ----------------
def build_group_indices(s: pd.Series):
    gi = s.groupby(s).indices
    gi.pop("", None)
    return {k: np.asarray(idx, dtype=np.int64) for k, idx in gi.items()}

def make_block_keys(df, cols):
    k1 = df[cols.zipc].str.cat(df[cols.city], sep="|").str.cat(df[cols.state], sep="|")
    k2 = df[cols.zipc].str.cat(df[cols.state], sep="|")
    k3 = df[cols.zipc]
    k4 = df[cols.city].str.cat(df[cols.state], sep="|")
    return [k1, k2, k3, k4]

# ---------------- Per-block ANN using precomputed embeddings ----------------
def ann_block_rows(q_idx, c_idx, drv, uc, cols, codes_d, codes_u, emb_d_all, emb_u_all,
                   thresh, k, small_cutoff, use_hn, gpu_ok):
    if len(q_idx)==0 or len(c_idx)==0:
        return None, None

    d_sub = drv.iloc[q_idx]
    u_sub = uc.iloc[c_idx]

    if use_hn:
        rows_out, matched_dlns = [], set()
        uc_grp = u_sub.groupby("_HN").indices
        for hn, d_pos in d_sub.groupby("_HN").indices.items():
            if hn not in uc_grp:
                continue
            d_sel = np.asarray(d_pos, dtype=np.int64)
            u_sel = np.asarray(uc_grp[hn], dtype=np.int64)
            r, m = _ann_block_core(
                q_idx[d_sel], c_idx[u_sel], drv, uc, cols,
                codes_d, codes_u, emb_d_all, emb_u_all,
                thresh, k, small_cutoff, gpu_ok
            )
            if r:
                rows_out.extend(r); matched_dlns.update(m)
        return rows_out, matched_dlns
    else:
        return _ann_block_core(
            q_idx, c_idx, drv, uc, cols,
            codes_d, codes_u, emb_d_all, emb_u_all,
            thresh, k, small_cutoff, gpu_ok
        )

def _ann_block_core(q_idx, c_idx, drv, uc, cols, codes_d, codes_u, emb_d_all, emb_u_all,
                    thresh, k, small_cutoff, gpu_ok):
    d_codes = codes_d[q_idx]
    u_codes = codes_u[c_idx]
    Ud = np.unique(d_codes); Uu = np.unique(u_codes)
    emb_d = emb_d_all[Ud]; emb_u = emb_u_all[Uu]
    if emb_d.size == 0 or emb_u.size == 0:
        return None, None

    topk = min(k, emb_u.shape[0])
    rows_out, matched_dlns = [], set()

    if emb_u.shape[0] <= small_cutoff:
        sims = emb_d @ emb_u.T
        idx = np.argpartition(-sims, kth=topk-1, axis=1)[:, :topk]
        part_scores = np.take_along_axis(sims, idx, axis=1)
        mask = part_scores >= thresh
        if not np.any(mask):
            return None, None

        code_to_dpos = {code: np.where(d_codes == code)[0] for code in Ud}
        code_to_upos = {code: np.where(u_codes == code)[0] for code in Uu}
        dln_arr = drv.iloc[q_idx][cols.dln].to_numpy()
        uid_arr = uc.iloc[c_idx]["UnclaimedUniqueID"].to_numpy()

        for i, code_d in enumerate(Ud):
            cols_i = idx[i, mask[i]]
            if cols_i.size == 0: 
                continue
            d_rows_local = code_to_dpos[code_d]
            dlns = dln_arr[d_rows_local]
            for jpos in cols_i:
                sc = float(sims[i, jpos])
                code_u = Uu[jpos]
                u_rows_local = code_to_upos[code_u]
                uids = uid_arr[u_rows_local]
                Dl = np.repeat(dlns, len(uids)); Ul = np.tile(uids, len(dlns))
                rows_out.extend(zip(Dl.tolist(), Ul.tolist(), [sc]*len(Dl)))
                matched_dlns.update(dlns.tolist())
        return rows_out, matched_dlns

    # FAISS FlatIP (GPU if available)
    if gpu_ok:
        res = faiss.StandardGpuResources()
        index = faiss.GpuIndexFlatIP(res, emb_u.shape[1])
    else:
        index = faiss.IndexFlatIP(emb_u.shape[1])
    index.add(emb_u)
    D, I = index.search(emb_d, topk)

    code_to_dpos = {code: np.where(d_codes == code)[0] for code in Ud}
    code_to_upos = {code: np.where(u_codes == code)[0] for code in Uu}
    dln_arr = drv.iloc[q_idx][cols.dln].to_numpy()
    uid_arr = uc.iloc[c_idx]["UnclaimedUniqueID"].to_numpy()

    for i, code_d in enumerate(Ud):
        sims = D[i]; ids = I[i]
        m = sims >= thresh
        if not np.any(m): 
            continue
        d_rows_local = code_to_dpos[code_d]
        dlns = dln_arr[d_rows_local]
        for jpos in ids[m]:
            code_u = Uu[jpos]
            u_rows_local = code_to_upos[code_u]
            uids = uid_arr[u_rows_local]
            Dl = np.repeat(dlns, len(uids)); Ul = np.tile(uids, len(dlns))
            rows_out.extend(zip(Dl.tolist(), Ul.tolist(), [float(s) for s in sims[m] for _ in range(len(dlns))][:len(Dl)]))
            matched_dlns.update(dlns.tolist())
    return rows_out, matched_dlns

# ---------------- Main ----------------
def main():
    args = parse_args()
    np.random.seed(args.seed)
    gpu_ok = torch.cuda.is_available() and faiss.get_num_gpus() > 0

    # -------- Load
    drv = pd.read_parquet(args.driver_parquet)
    uc  = pd.read_parquet(args.unclaimed_parquet)

    # Column detection
    class Cols: pass
    cols = Cols()
    cols.dln   = pick_col(drv, ["DLN"])
    cols.city  = pick_col(drv, ["AddressCity","Res_City","City"])
    cols.state = pick_col(drv, ["AddressState","Res_State","State"])
    cols.zipc  = pick_col(drv, ["AddressZip","Zip5","Res_ZipCodeFull"], required=True)

    # unify names on both sides
    drv.rename(columns={cols.city:"__CITY", cols.state:"__STATE", cols.zipc:"__ZIP"}, inplace=True)
    cols.city="__CITY"; cols.state="__STATE"; cols.zipc="__ZIP"
    uc_city  = pick_col(uc, ["AddressCity","City"])
    uc_state = pick_col(uc, ["AddressState","State"])
    uc_zip   = pick_col(uc, ["AddressZip","Zip5"])
    uc.rename(columns={uc_city:"__CITY", uc_state:"__STATE", uc_zip:"__ZIP"}, inplace=True)

    # Normalize block fields and street text
    for df in (drv, uc):
        df["__CITY"]  = norm_key(df["__CITY"])
        df["__STATE"] = norm_key(df["__STATE"]).str.replace(r"[^A-Z]", "", regex=True).str[:2]
        df["__ZIP"]   = safe_zip5(df["__ZIP"])
        df["AddressStreet"] = norm_key(df["AddressStreet"])

    # filter placeholder streets
    drv = drv[~drv["AddressStreet"].str.match(PLACEHOLDER_ADDR, na=False)]
    uc  = uc[ ~uc["AddressStreet"].str.match(PLACEHOLDER_ADDR, na=False)]

    # house numbers for optional sub-block
    drv["_HN"] = house_num(drv["AddressStreet"])
    uc["_HN"]  = house_num(uc["AddressStreet"])

    if len(drv)==0 or len(uc)==0:
        pd.DataFrame(columns=["DLN","UnclaimedUniqueID","AddressScore"]).to_parquet(args.out_parquet, index=False)
        print(f"drv={len(drv)}, uc={len(uc)} → no data; wrote empty {args.out_parquet}")
        return

    # -------- Tier-1 exact join (vectorized)
    exact_df = exact_join_tier1(drv, uc, cols)
    matched_dlns = set(exact_df["DLN"].unique()) if not exact_df.empty else set()

    # -------- Global de-dup & embedding (done ONCE)
    model = load_model()
    codes_d, uniques_d, emb_d_all = dedup_and_embed(model, drv["AddressStreet"], batch=args.embed_batch)
    codes_u, uniques_u, emb_u_all = dedup_and_embed(model, uc ["AddressStreet"], batch=args.embed_batch)

    # -------- Build block keys once
    drv_keys = make_block_keys(drv, cols)
    uc_keys  = make_block_keys(uc,  cols)
    uc_maps  = [build_group_indices(k) for k in uc_keys]

    # -------- Iterate tiers with ANN per block
    kept_rows = []
    if not exact_df.empty:
        kept_rows.extend(exact_df.itertuples(index=False, name=None))

    matched_mask = np.isin(drv[cols.dln].to_numpy(), list(matched_dlns)) if args.stop_at_first_tier_match else np.zeros(len(drv), dtype=bool)

    total_blocks = 0
    for tier, (drv_key_ser, uc_map) in enumerate(zip(drv_keys, uc_maps), start=1):
        drv_grp = drv_key_ser.groupby(drv_key_ser).indices
        drv_grp.pop("", None)
        for key, q_idx_all in drv_grp.items():
            if key == "" or key not in uc_map:
                continue
            total_blocks += 1
            q_idx_all = np.asarray(q_idx_all, dtype=np.int64)
            c_idx = uc_map[key]

            if args.stop_at_first_tier_match:
                q_idx = q_idx_all[~matched_mask[q_idx_all]]
                if q_idx.size == 0:
                    continue
            else:
                q_idx = q_idx_all

            rows, matched = ann_block_rows(
                q_idx, c_idx, drv, uc, cols,
                codes_d, codes_u, emb_d_all, emb_u_all,
                thresh=args.thresh, k=args.k,
                small_cutoff=args.small_block_cutoff,
                use_hn=args.use_house_number,
                gpu_ok=gpu_ok
            )
            if rows:
                kept_rows.extend(rows)
                if args.stop_at_first_tier_match and matched:
                    mm = drv[cols.dln].isin(matched).to_numpy()
                    matched_mask |= mm

    # -------- Finalize
    out = pd.DataFrame(kept_rows, columns=["DLN","UnclaimedUniqueID","AddressScore"]).drop_duplicates()
    out.to_parquet(args.out_parquet, index=False)
    print(f"drv={len(drv):,}, uc={len(uc):,}, blocks={total_blocks:,}, exact={(0 if exact_df is None else len(exact_df)):,}, kept={len(out):,} -> {args.out_parquet}")

if __name__ == "__main__":
    # Jupyter-safe: parse_known_args handles -f <kernel.json>
    main()


/var/tmp/ipykernel_4641/161579523.py:268: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uc["_HN"]  = house_num(uc["AddressStreet"])
/var/tmp/ipykernel_4641/161579523.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uc[key]  = uc[cols.zipc].str.cat(uc[cols.city],  sep="|").str.cat(uc[cols.state],  sep="|")
/var/tmp/ipykernel_4641/161579523.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pa

drv=5,937,860, uc=679,036, blocks=16,062, exact=972,942, kept=9,341,532 -> address_ann_matches.parquet


### Joining both addresses and names parquets

In [5]:
#!/usr/bin/env python3
"""
step5_join_label_clean.py
- Joins name + address ANN outputs
- Clips scores to [0,1]
- Recomputes MatchType from thresholds (source of truth)
- Enforces Address-Only acceptance (no placeholders, valid ZIPs)
- Dedupe to one row per (DriverUniqueID, UnclaimedUniqueID)
- Attaches ValueBucket (Over $100 / Under $100 / Unknown)
- Writes clean final parquet (+ optional CSVs for Both and full joined)

Inputs:
  name_ann_matches.parquet      (DLN, UnclaimedUniqueID, NameScore)
  address_ann_matches.parquet   (DLN, UnclaimedUniqueID, AddressScore)
  driver_dl_for_ann.parquet     (DLN, FullName, FullAddress, Zip5 [optional])
  unclaimed_for_ann.parquet     (UnclaimedUniqueID, FullName_cleaned, FullAddress_cleaned,
                                 Zip5 [optional], PropertyTypeCodeDescription)

Outputs:
  final_matches.parquet (clean)
  [optional] final_matches_both.csv(.gz)
  [optional] final_matches_joined.csv(.gz)

Usage (defaults shown):
  python step5_join_label_clean.py \
    --name_parquet name_ann_matches.parquet \
    --addr_parquet address_ann_matches.parquet \
    --driver_parquet driver_dl_for_ann.parquet \
    --unclaimed_parquet unclaimed_for_ann.parquet \
    --out_parquet final_matches.parquet \
    --both_csv final_matches_both.csv \
    --joined_csv final_matches_joined.csv \
    [--gzip_csv]
"""

import argparse
import re
import numpy as np
import pandas as pd
from pathlib import Path
from google.cloud import bigquery

NAME_THRESH = 0.90
ADDR_THRESH = 0.95
PLACEHOLDER_ADDR = r"^(|N/?A|NONE|UNKNOWN|UNK|NO ADDRESS|ADDRESS UNKNOWN|\.)$"


def quick_zip(series: pd.Series) -> pd.Series:
    """Extract a 5-digit ZIP from a free-text address (best-effort)."""
    return series.astype(str).str.extract(r"(\b\d{5}\b)", expand=False).fillna("")


def accept_address_only(row) -> bool:
    """Treasury rule: Address-Only must not contain placeholders and must have valid ZIPs on both sides."""
    d_addr = str(row.get("DriverAddress", ""))
    u_addr = str(row.get("UnclaimedAddress", ""))
    d_zip  = str(row.get("DriverZip5", ""))
    u_zip  = str(row.get("UnclaimedZip5",""))
    if re.match(PLACEHOLDER_ADDR, d_addr) or re.match(PLACEHOLDER_ADDR, u_addr):
        return False
    if d_zip in ("", "00000") or u_zip in ("", "00000"):
        return False
    return True


def write_csv(df: pd.DataFrame, path: str, gzip: bool = False):
    if not path:
        return
    p = Path(path)
    if gzip and not p.suffix.endswith(".gz"):
        p = p.with_suffix(p.suffix + ".gz")
    df.to_csv(p, index=False, compression=("gzip" if gzip else None))
    print(f"✅ Wrote {len(df):,} rows -> {p.name}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name_parquet",      default="name_ann_matches.parquet")
    ap.add_argument("--addr_parquet",      default="address_ann_matches.parquet")
    ap.add_argument("--driver_parquet",    default="driver_dl_for_ann.parquet")
    ap.add_argument("--unclaimed_parquet", default="unclaimed_for_ann.parquet")
    ap.add_argument("--out_parquet",       default="final_matches.parquet")
    ap.add_argument("--both_csv",          default="")       # optional
    ap.add_argument("--joined_csv",        default="")       # optional full joined CSV
    ap.add_argument("--gzip_csv",          action="store_true", default=False)
    ap.add_argument("--keep_dln",          action="store_true", default=False,
                    help="Keep DLN column in final if present (otherwise drop when == DriverUniqueID).")
    args, _ = ap.parse_known_args()

    # ---------- Load ANN outputs (max-score aggregate per pair) ----------
    name_df = pd.read_parquet(args.name_parquet) if args.name_parquet else pd.DataFrame(columns=["DLN","UnclaimedUniqueID","NameScore"])
    addr_df = pd.read_parquet(args.addr_parquet) if args.addr_parquet else pd.DataFrame(columns=["DLN","UnclaimedUniqueID","AddressScore"])

    for df in (name_df, addr_df):
        if "DLN" in df.columns:
            df["DLN"] = df["DLN"].astype(str)
        if "UnclaimedUniqueID" in df.columns:
            df["UnclaimedUniqueID"] = df["UnclaimedUniqueID"].astype(str)

    if len(name_df):
        name_agg = (
            name_df.groupby(["DLN","UnclaimedUniqueID"], as_index=False)
                   .agg(NameScore=("NameScore","max"))
        )
    else:
        name_agg = pd.DataFrame(columns=["DLN","UnclaimedUniqueID","NameScore"])

    if len(addr_df):
        addr_agg = (
            addr_df.groupby(["DLN","UnclaimedUniqueID"], as_index=False)
                   .agg(AddressScore=("AddressScore","max"))
        )
    else:
        addr_agg = pd.DataFrame(columns=["DLN","UnclaimedUniqueID","AddressScore"])

    pairs = name_agg.merge(addr_agg, on=["DLN","UnclaimedUniqueID"], how="outer", copy=False)

    # Coerce scores to numeric now; we'll clip later
    for c in ("NameScore","AddressScore"):
        if c in pairs.columns:
            pairs[c] = pd.to_numeric(pairs[c], errors="coerce")

    # ---------- Attach readable fields + zips from sources ----------
    drv = pd.read_parquet(args.driver_parquet, columns=None).copy()
    uc  = pd.read_parquet(args.unclaimed_parquet, columns=None).copy()

    drv["DLN"] = drv["DLN"].astype(str)
    uc["UnclaimedUniqueID"] = uc["UnclaimedUniqueID"].astype(str)

    # Build 1-row per key for lookup
    drv_one = (
        drv.sort_values("DLN")
           .drop_duplicates("DLN", keep="first")
           .rename(columns={
               "FullName":   "DriverName",
               "FullAddress":"DriverAddress",
               "Zip5":       "DriverZip5"
           })
    )

    uc_one  = (
        uc.sort_values("UnclaimedUniqueID")
          .drop_duplicates("UnclaimedUniqueID", keep="first")
          .rename(columns={
              "FullName_cleaned":    "UnclaimedName",
              "FullAddress_cleaned": "UnclaimedAddress",
              "Zip5":                "UnclaimedZip5"
          })
    )

    out = (
        pairs
        .merge(
            drv_one[["DLN","DriverName","DriverAddress","DriverZip5"]].drop_duplicates("DLN"),
            on="DLN", how="left", copy=False
        )
        .merge(
            uc_one[["UnclaimedUniqueID","UnclaimedName","UnclaimedAddress",
                    "UnclaimedZip5","PropertyTypeCodeDescription"]]
            .drop_duplicates("UnclaimedUniqueID"),
            on="UnclaimedUniqueID", how="left", copy=False
        )
    )

    # -----------------------------------------------------------
    # ATTACH VALUE BUCKET (Over $100 / Under $100 / Unknown)
    # from original treasury PERSON table
    # -----------------------------------------------------------
    client = bigquery.Client()

    sql = """
    SELECT
      CONCAT(CAST(PropertyID AS STRING), '_', CAST(OwnerID AS STRING)) AS UnclaimedUniqueID,
      Value
    FROM `omes-datascience-sbx.Unclaimed_Property_Analysis.unclaimed_property_new_data_20250730`
    WHERE OwnerType = 'PERSON'
    """
    value_lookup = client.query(sql).to_dataframe()

    # Normalize the bucket to 3 values
    value_lookup["ValueBucket"] = (
        value_lookup["Value"]
        .fillna("")
        .replace({"": "Unknown"})
    )

    # Merge into the final output
    out = out.merge(
        value_lookup[["UnclaimedUniqueID", "ValueBucket"]],
        on="UnclaimedUniqueID",
        how="left"
    )

    # DriverUniqueID = DLN (system of record)
    out["DriverUniqueID"] = out["DLN"].astype(str)

    # Ensure ZIPs exist (derive if not in source)
    if "DriverZip5" not in out.columns or out["DriverZip5"].isna().all():
        out["DriverZip5"] = quick_zip(out["DriverAddress"])
    if "UnclaimedZip5" not in out.columns or out["UnclaimedZip5"].isna().all():
        out["UnclaimedZip5"] = quick_zip(out["UnclaimedAddress"])

    # ---------- Clip scores, recompute MatchType from thresholds ----------
    for c in ("NameScore","AddressScore"):
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").clip(0, 1)

    has_name = out["NameScore"].notna() & (out["NameScore"] >= NAME_THRESH)
    has_addr = out["AddressScore"].notna() & (out["AddressScore"] >= ADDR_THRESH)
    out["MatchType"] = np.select(
        [has_name & has_addr, has_addr & ~has_name, has_name & ~has_addr],
        ["Both", "Address Only", "Name Only"],
        default="(none)"
    )

    # ---------- Enforce Address-Only acceptance rule ----------
    addr_only_mask = out["MatchType"].eq("Address Only")
    if addr_only_mask.any():
        accepted = out.loc[addr_only_mask].apply(accept_address_only, axis=1)
        dropped = int((~accepted).sum())
        if dropped:
            out = pd.concat(
                [out.loc[~addr_only_mask], out.loc[addr_only_mask][accepted.values]],
                ignore_index=True
            )
        print(f"Address-Only rows dropped by acceptance filter: {dropped:,}")

    # ---------- Dedupe: one row per (DriverUniqueID, UnclaimedUniqueID) ----------
    # Preference: Both > Name Only > Address Only, then highest NameScore, then AddressScore
    out["pref_match"] = out["MatchType"].map(
        {"Both": 2, "Name Only": 1, "Address Only": 0, "(none)": -1}
    ).fillna(-1)
    out = (
        out.sort_values(by=["pref_match","NameScore","AddressScore"],
                        ascending=[False, False, False])
           .drop_duplicates(subset=["DriverUniqueID","UnclaimedUniqueID"], keep="first")
           .drop(columns=["pref_match"])
           .reset_index(drop=True)
    )

    # ---------- Drop DLN if identical to DriverUniqueID (unless keep_dln is requested) ----------
    if not args.keep_dln and "DLN" in out.columns:
        same = (out["DLN"].astype(str) == out["DriverUniqueID"].astype(str)).all()
        if same:
            out = out.drop(columns=["DLN"])

    # ---------- Round scores for CSV readability ----------
    for c in ("NameScore","AddressScore"):
        if c in out.columns:
            out[c] = out[c].round(2)

    # ---------- Final column order ----------
    want = [
        "DLN",  # only present if --keep_dln or not identical
        "DriverUniqueID",
        "DriverName",
        "DriverAddress",
        "MatchType",
        "NameScore",
        "AddressScore",
        "UnclaimedUniqueID",
        "UnclaimedName",
        "UnclaimedAddress",
        "PropertyTypeCodeDescription",
        "ValueBucket",
    ]
    cols = [c for c in want if c in out.columns] + [c for c in out.columns if c not in want]
    out = out[cols]

    # ---------- QC summary ----------
    N = len(out)
    ct = out["MatchType"].value_counts()
    n_both = int(ct.get("Both", 0))
    n_name = int(ct.get("Name Only", 0))
    n_addr = int(ct.get("Address Only", 0))

    print("=== Final Summary ===")
    print(f"Rows written: {N:,}")
    print(f"  Both:         {n_both:,} ({(n_both/N*100 if N else 0):.2f}%)")
    print(f"  Name Only:    {n_name:,} ({(n_name/N*100 if N else 0):.2f}%)")
    print(f"  Address Only: {n_addr:,} ({(n_addr/N*100 if N else 0):.2f}%)")

    # ---------- NEW: Value bucket stats (4.1 / 4.2 / 4.3) ----------

    # 4.1 – all matches
    bucket_counts_all = (
        out.groupby("ValueBucket")["UnclaimedUniqueID"]
           .nunique()
           .rename("PropertyCount")
           .reset_index()
    )

    print("\n=== Matched properties by value bucket (ALL MatchTypes) ===")
    print(bucket_counts_all.to_string(index=False))

    # 4.2 – only high-confidence “Both”
    both_mask = out["MatchType"].eq("Both")
    bucket_counts_both = (
        out.loc[both_mask]
           .groupby("ValueBucket")["UnclaimedUniqueID"]
           .nunique()
           .rename("PropertyCount")
           .reset_index()
    )

    print("\n=== High-confidence BOTH matches by value bucket ===")
    print(bucket_counts_both.to_string(index=False))

    # 4.3 – add percentages for “Both”
    total_both = bucket_counts_both["PropertyCount"].sum()
    if total_both > 0:
        bucket_counts_both["Percent"] = (
            bucket_counts_both["PropertyCount"] / total_both * 100
        ).round(2)
    else:
        bucket_counts_both["Percent"] = 0.0

    print("\n=== BOTH matches by value bucket (with percentages) ===")
    print(bucket_counts_both.to_string(index=False))

    # ---------- Save outputs ----------
    out.to_parquet(args.out_parquet, index=False)
    print(f"✅ Wrote {len(out):,} rows -> {args.out_parquet}")

    # BOTH CSV
    if args.both_csv:
        both = out[out["MatchType"].eq("Both")].copy()
        preferred_cols = [
            "DriverUniqueID","DriverName","DriverAddress",
            "UnclaimedUniqueID","UnclaimedName","UnclaimedAddress",
            "PropertyTypeCodeDescription","ValueBucket",
            "NameScore","AddressScore","MatchType"
        ]
        both = both[[c for c in preferred_cols if c in both.columns] +
                    [c for c in both.columns if c not in preferred_cols]]
        write_csv(both, args.both_csv, gzip=args.gzip_csv)

    # FULL JOINED CSV
    if args.joined_csv:
        preferred_cols = [
            "DriverUniqueID","DriverName","DriverAddress",
            "UnclaimedUniqueID","UnclaimedName","UnclaimedAddress",
            "PropertyTypeCodeDescription","ValueBucket",
            "NameScore","AddressScore","MatchType"
        ]
        joined = out[[c for c in preferred_cols if c in out.columns] +
                     [c for c in out.columns if c not in preferred_cols]]
        write_csv(joined, args.joined_csv, gzip=args.gzip_csv)


if __name__ == "__main__":
    main()


Address-Only rows dropped by acceptance filter: 11,991
=== Final Summary ===
Rows written: 6,727,204
  Both:         9,610 (0.14%)
  Name Only:    844,664 (12.56%)
  Address Only: 5,872,928 (87.30%)

=== Matched properties by value bucket (ALL MatchTypes) ===
ValueBucket  PropertyCount
  Over $100         211991
 Under $100         183206
    Unknown            345

=== High-confidence BOTH matches by value bucket ===
ValueBucket  PropertyCount
  Over $100           2873
 Under $100           4701

=== BOTH matches by value bucket (with percentages) ===
ValueBucket  PropertyCount  Percent
  Over $100           2873    37.93
 Under $100           4701    62.07
✅ Wrote 6,727,204 rows -> final_matches.parquet


In [6]:
# --- Duplicate checks (robust) ---
import os
import pandas as pd

# 1) Load the final output (or reuse if you already have it in memory)
FINAL_PARQUET = "final_matches.parquet"
if 'final' in globals():
    final_df = final.copy()
elif os.path.exists(FINAL_PARQUET):
    final_df = pd.read_parquet(FINAL_PARQUET)
else:
    raise FileNotFoundError("Couldn't find final_matches.parquet and variable `final` not in scope.")

# 2) Subset: only 'Both' if you want a focused check (optional)
both_df = final_df[final_df["MatchType"].eq("Both")].copy()

# 3) Helper to pick available keys
def pick_key(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def dup_report(
    df,
    name,
    driver_candidates=("DriverUniqueID","DLN"),
    uid_candidate="UnclaimedUniqueID",
    save_csv=True,
    prefix="dups"
):
    print(f"\n=== DUPLICATE REPORT: {name} ===")
    print(f"rows: {len(df):,}")

    # 0) exact duplicate rows (all columns identical)
    if len(df):
        exact_mask = df.duplicated(keep=False)
        exact_dups = df.loc[exact_mask]
        print(f"exact-duplicate rows (all cols identical): {len(exact_dups):,}")
        if save_csv and len(exact_dups):
            exact_dups.to_csv(f"{prefix}_{name}_exact.csv", index=False)
    else:
        print("empty dataframe; skipping.")
        return

    driver_key = pick_key(df, driver_candidates)
    uid_key    = uid_candidate if uid_candidate in df.columns else None

    if driver_key:
        vc_driver = df[driver_key].value_counts()
        multi_driver_ids = vc_driver[vc_driver > 1].index
        multi_driver = df[df[driver_key].isin(multi_driver_ids)]
        print(f"drivers with >1 row ({driver_key}): {len(multi_driver_ids):,} (rows: {len(multi_driver):,})")
        if save_csv and len(multi_driver):
            multi_driver.sort_values([driver_key]).to_csv(f"{prefix}_{name}_by_driver.csv", index=False)
    else:
        print("no driver key found; skipped per-driver check.")

    if uid_key:
        vc_uid = df[uid_key].value_counts()
        multi_uid_ids = vc_uid[vc_uid > 1].index
        multi_uid = df[df[uid_key].isin(multi_uid_ids)]
        print(f"unclaimed IDs with >1 row ({uid_key}): {len(multi_uid_ids):,} (rows: {len(multi_uid):,})")
        if save_csv and len(multi_uid):
            multi_uid.sort_values([uid_key]).to_csv(f"{prefix}_{name}_by_unclaimed.csv", index=False)
    else:
        print("no UnclaimedUniqueID column; skipped per-unclaimed check.")

    if driver_key and uid_key:
        pair_mask = df.duplicated(subset=[driver_key, uid_key], keep=False)
        pair_dups = df.loc[pair_mask].sort_values([driver_key, uid_key])
        print(f"duplicate pairs ({driver_key}, {uid_key}): {len(pair_dups):,}")
        if save_csv and len(pair_dups):
            pair_dups.to_csv(f"{prefix}_{name}_pairs.csv", index=False)
    else:
        print("missing one or both pair keys; skipped pair duplicate check.")

# 4) Run reports
dup_report(final_df, "final_all", prefix="dups")
dup_report(both_df,  "final_both", prefix="dups")

# 5) Hard fail if there are duplicate pairs in the shipped artifact
driver_key = "DriverUniqueID" if "DriverUniqueID" in final_df.columns else ("DLN" if "DLN" in final_df.columns else None)
uid_key    = "UnclaimedUniqueID" if "UnclaimedUniqueID" in final_df.columns else None

if driver_key and uid_key:
    dup_pair_ct = final_df.duplicated(subset=[driver_key, uid_key], keep=False).sum()
    assert dup_pair_ct == 0, f"Duplicate pairs found in final output: {dup_pair_ct}"
    print("✅ No duplicate pairs in final output.")
else:
    print("⚠️ Skipped hard-fail pair check (missing keys).")



=== DUPLICATE REPORT: final_all ===
rows: 6,727,204
exact-duplicate rows (all cols identical): 0
drivers with >1 row (DriverUniqueID): 798,230 (rows: 6,067,019)
unclaimed IDs with >1 row (UnclaimedUniqueID): 383,032 (rows: 6,639,580)
duplicate pairs (DriverUniqueID, UnclaimedUniqueID): 0

=== DUPLICATE REPORT: final_both ===
rows: 9,610
exact-duplicate rows (all cols identical): 0
drivers with >1 row (DriverUniqueID): 791 (rows: 7,488)
unclaimed IDs with >1 row (UnclaimedUniqueID): 599 (rows: 1,204)
duplicate pairs (DriverUniqueID, UnclaimedUniqueID): 0
✅ No duplicate pairs in final output.


In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""
### Duplicate reports

- **`dups_final_both_exact.csv`**  
  - **What:** Rows that are *identical across all columns*.  
  - **Cause:** Same link surfaced via multiple paths / repeated ANN candidates.  
  - **Action:** Safe to **drop**.

- **`dups_final_both_by_driver.csv`**  
  - **What:** Drivers (**DriverUniqueID**) appearing in **>1 row** (all their rows included).  
  - **Meaning:** One driver matched to multiple unclaimed properties — often **legit**.  
  - **Action:** Keep if one-to-many is allowed; otherwise **keep the best per driver** (highest `NameScore`, then `AddressScore`).

- **`dups_final_both_by_unclaimed.csv`**  
  - **What:** Unclaimed properties (**UnclaimedUniqueID**) appearing in **>1 row**.  
  - **Meaning:** One property matched to multiple drivers — **needs review/tie-break**.  
  - **Action:** **Resolve to a single owner per property** (highest `NameScore`, then `AddressScore`).

- **`dups_final_both_pairs.csv`**  
  - **What:** Duplicate **(DriverUniqueID, UnclaimedUniqueID)** pairs.  
  - **Meaning:** True pairwise duplicates.  
  - **Action:** **Dedupe to one row per pair** .
"""))


In [ ]:
final_df.shape, both_df.shape, final_df.columns.tolist()


### dropping DLN and rounding scores between 0 and 1

In [13]:
### dropping DLN and rounding scores between 0 and 1 + Weighted Score
import pandas as pd
import numpy as np

IN_PARQUET = "final_matches.parquet"
OUT_CSV    = "final_matches_both.csv"

# Load if not already in memory
try:
    final_df
except NameError:
    final_df = pd.read_parquet(IN_PARQUET)

# Keep only 'Both'
both_df = final_df[final_df["MatchType"].eq("Both")].copy()

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # Ensure we have a driver key; prefer DriverUniqueID, else use DLN
    if "DriverUniqueID" not in df.columns and "DLN" in df.columns:
        df = df.rename(columns={"DLN": "DriverUniqueID"})
    # Drop DLN if still present
    if "DLN" in df.columns:
        df = df.drop(columns=["DLN"])

    # Scores → numeric, clip [0,1], round 2 decimals
    for c in ("NameScore", "AddressScore"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).clip(0, 1).round(2)

    # ---------- Weighted Score ----------
    # Weighted average: equal weight 0.5 / 0.5
    df["WeightedScore"] = (
        0.5 * df["NameScore"] +
        0.5 * df["AddressScore"]
    ).round(2)

    return df

# Clean both dataframes
final_df = clean_df(final_df)
both_df  = clean_df(both_df)

# Drop ZIP columns if present
for col in ["DriverZip5", "UnclaimedZip5"]:
    if col in both_df.columns:
        both_df = both_df.drop(columns=[col])
    if col in final_df.columns:
        final_df = final_df.drop(columns=[col])

# Quick checks
print("final_df shape:", final_df.shape)
print("both_df  shape:", both_df.shape)

# Reorder columns for CSV
col_order = [
    "DriverUniqueID", "DriverName", "DriverAddress",
    "UnclaimedUniqueID", "UnclaimedName", "UnclaimedAddress",
    "PropertyTypeCodeDescription",
    "NameScore", "AddressScore", "WeightedScore"
]

both_df = both_df[
    [c for c in col_order if c in both_df.columns] +
    [c for c in both_df.columns if c not in col_order]
]

# Save CSV
both_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(both_df):,} rows -> {OUT_CSV}")


final_df shape: (6727204, 12)
both_df  shape: (9610, 12)
Wrote 9,610 rows -> final_matches_both.csv


### Data Validation and join check

In [8]:
#!/usr/bin/env python3
# qa_verify_joins_exact.py — Verify final joins are exact copies of source fields
# - Ensures DriverName/DriverAddress match driver source (exact string equality)
# - Ensures UnclaimedName/UnclaimedAddress/PropertyTypeCodeDescription match unclaimed source (exact string equality)
# - Flags any missing IDs or conflicting duplicates in sources
# - Writes CSVs with mismatches for debugging

import argparse
from pathlib import Path
import pandas as pd

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument(
        "--final_parquet",
        default="final_matches.parquet",
        help="Final joined output (cleaned) to validate."
    )
    ap.add_argument(
        "--driver_parquet",
        default="driver_dl_for_ann.parquet",
        help="Driver source parquet."
    )
    ap.add_argument(
        "--unclaimed_parquet",
        default="unclaimed_for_ann.parquet",
        help="Unclaimed source parquet."
    )
    ap.add_argument(
        "--outdir",
        default="qa_reports_join_exact",
        help="Directory to write mismatch/diagnostic CSVs."
    )
    # If your final file uses DLN as the driver key instead of DriverUniqueID, set this accordingly.
    ap.add_argument(
        "--driver_key_in_final",
        default="DriverUniqueID",
        help="Key column name in FINAL that maps to driver DLN (usually 'DriverUniqueID')."
    )
    args, _ = ap.parse_known_args()  # jupyter-safe
    return args

def ensure_cols(df, need, label):
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"Missing columns in {label}: {miss}")

def uniqueness_conflicts(df, key, cols):
    """
    Returns a dataframe of conflicting duplicates if any:
    - Multiple rows with the same key but different values in 'cols'.
    Otherwise returns empty df.
    """
    vc = df[key].value_counts()
    dup_keys = vc[vc > 1].index
    if len(dup_keys) == 0:
        return pd.DataFrame()
    distinct = df.loc[df[key].isin(dup_keys), [key] + cols].drop_duplicates()
    bad_keys = distinct.groupby(key).size()
    bad_keys = bad_keys[bad_keys > 1].index
    return distinct[distinct[key].isin(bad_keys)].sort_values(key)

def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(exist_ok=True)

    # ---------- Load ----------
    fin = pd.read_parquet(args.final_parquet)
    drv = pd.read_parquet(args.driver_parquet)
    uc  = pd.read_parquet(args.unclaimed_parquet)

    # ---------- Expected schemas ----------
    ensure_cols(drv, ["DLN", "FullName", "FullAddress"], "driver source")
    ensure_cols(
        uc,
        ["UnclaimedUniqueID", "FullName_cleaned", "FullAddress_cleaned", "PropertyTypeCodeDescription"],
        "unclaimed source"
    )

    # In final, we validate these columns (must exist exactly)
    final_required = [
        args.driver_key_in_final, "UnclaimedUniqueID",
        "DriverName", "DriverAddress",
        "UnclaimedName", "UnclaimedAddress",
        "PropertyTypeCodeDescription",
    ]
    ensure_cols(fin, final_required, "final output")

    # ---------- Normalize key types (string keys for exact merge) ----------
    fin[args.driver_key_in_final] = fin[args.driver_key_in_final].astype(str)
    fin["UnclaimedUniqueID"]      = fin["UnclaimedUniqueID"].astype(str)
    drv["DLN"]                    = drv["DLN"].astype(str)
    uc["UnclaimedUniqueID"]       = uc["UnclaimedUniqueID"].astype(str)

    # ---------- Source uniqueness & conflicts ----------
    drv_conf = uniqueness_conflicts(drv, "DLN", ["FullName", "FullAddress"])
    if not drv_conf.empty:
        p = outdir / "source_driver_conflicting_duplicates.csv"
        drv_conf.to_csv(p, index=False)
        print(f"❌ Driver source has keys with conflicting values → {p}")
    else:
        print("✅ Driver source unique per DLN (or duplicates have identical values).")

    uc_conf = uniqueness_conflicts(
        uc,
        "UnclaimedUniqueID",
        ["FullName_cleaned", "FullAddress_cleaned", "PropertyTypeCodeDescription"]
    )
    if not uc_conf.empty:
        p = outdir / "source_unclaimed_conflicting_duplicates.csv"
        uc_conf.to_csv(p, index=False)
        print(f"❌ Unclaimed source has keys with conflicting values → {p}")
    else:
        print("✅ Unclaimed source unique per UnclaimedUniqueID (or duplicates have identical values).")

    # ---------- Build one-row-per-key lookups (first occurrence OK if no conflicts) ----------
    drv_one = (
        drv.drop_duplicates("DLN", keep="first")
           .rename(columns={"FullName": "FullName_src", "FullAddress": "FullAddress_src"})
    )
    uc_one  = (
        uc.drop_duplicates("UnclaimedUniqueID", keep="first")
          .rename(columns={
              "FullName_cleaned": "UnclaimedName_src",
              "FullAddress_cleaned": "UnclaimedAddress_src",
              "PropertyTypeCodeDescription": "PropertyTypeCodeDescription_src"
          })
    )

    # ---------- Attach source columns to final for exact comparisons ----------
    chk = fin.merge(
        drv_one[["DLN", "FullName_src", "FullAddress_src"]],
        left_on=args.driver_key_in_final,
        right_on="DLN",
        how="left",
        validate="many_to_one"
    )
    chk = chk.merge(
        uc_one[["UnclaimedUniqueID", "UnclaimedName_src", "UnclaimedAddress_src", "PropertyTypeCodeDescription_src"]],
        on="UnclaimedUniqueID",
        how="left",
        validate="many_to_one"
    )

    # ---------- Missing ID checks ----------
    missing_driver = chk[chk["DLN"].isna()]
    if len(missing_driver):
        p = outdir / "missing_driver_ids_in_source.csv"
        missing_driver.to_csv(p, index=False)
        print(f"❌ {len(missing_driver):,} final rows have driver key not found in driver source → {p}")
    else:
        print("✅ All driver keys in final exist in driver source.")

    missing_uc = chk[
        chk["UnclaimedName_src"].isna()
        & chk["UnclaimedAddress_src"].isna()
        & chk["PropertyTypeCodeDescription_src"].isna()
    ]
    if len(missing_uc):
        p = outdir / "missing_unclaimed_ids_in_source.csv"
        missing_uc.to_csv(p, index=False)
        print(f"❌ {len(missing_uc):,} final rows have UnclaimedUniqueID not found in unclaimed source → {p}")
    else:
        print("✅ All UnclaimedUniqueID values in final exist in unclaimed source.")

    # ---------- Exact equality checks (no normalization) ----------
    # Driver side
    drv_name_mismatch = chk[chk["DriverName"]    != chk["FullName_src"]]
    drv_addr_mismatch = chk[chk["DriverAddress"] != chk["FullAddress_src"]]

    # Unclaimed side
    uc_name_mismatch  = chk[chk["UnclaimedName"]    != chk["UnclaimedName_src"]]
    uc_addr_mismatch  = chk[chk["UnclaimedAddress"] != chk["UnclaimedAddress_src"]]
    ptype_mismatch    = chk[chk["PropertyTypeCodeDescription"] != chk["PropertyTypeCodeDescription_src"]]

    def export(df, fname, label, keep_cols):
        if len(df):
            outp = outdir / fname
            df[keep_cols].to_csv(outp, index=False)
            print(f"❌ {label}: {len(df):,} mismatch rows → {outp}")
        else:
            print(f"✅ {label}: exact match")

    # Base columns always included
    keep_cols = [
        args.driver_key_in_final, "UnclaimedUniqueID",
        "DriverName", "FullName_src",
        "DriverAddress", "FullAddress_src",
        "UnclaimedName", "UnclaimedName_src",
        "UnclaimedAddress", "UnclaimedAddress_src",
        "PropertyTypeCodeDescription", "PropertyTypeCodeDescription_src",
    ]

    # Optionally include ValueBucket / WeightedScore if present for context
    for extra in ("ValueBucket", "WeightedScore"):
        if extra in chk.columns and extra not in keep_cols:
            keep_cols.append(extra)

    export(drv_name_mismatch, "mismatch_driver_name.csv", "DriverName equals driver FullName_src", keep_cols)
    export(drv_addr_mismatch, "mismatch_driver_address.csv", "DriverAddress equals driver FullAddress_src", keep_cols)
    export(uc_name_mismatch,  "mismatch_unclaimed_name.csv", "UnclaimedName equals unclaimed FullName_cleaned", keep_cols)
    export(uc_addr_mismatch,  "mismatch_unclaimed_address.csv", "UnclaimedAddress equals unclaimed FullAddress_cleaned", keep_cols)
    export(ptype_mismatch,    "mismatch_property_type_desc.csv", "PropertyTypeCodeDescription equals unclaimed source", keep_cols)

    issues = sum(len(x) for x in [
        missing_driver, missing_uc,
        drv_name_mismatch, drv_addr_mismatch,
        uc_name_mismatch, uc_addr_mismatch, ptype_mismatch
    ])

    print("\n=== Join Verification Summary ===")
    print(f"Final rows checked: {len(fin):,}")
    print(f"Driver key missing in source: {len(missing_driver):,}")
    print(f"Unclaimed key missing in source: {len(missing_uc):,}")
    print(f"DriverName mismatches: {len(drv_name_mismatch):,}")
    print(f"DriverAddress mismatches: {len(drv_addr_mismatch):,}")
    print(f"UnclaimedName mismatches: {len(uc_name_mismatch):,}")
    print(f"UnclaimedAddress mismatches: {len(uc_addr_mismatch):,}")
    print(f"PropertyTypeCodeDescription mismatches: {len(ptype_mismatch):,}")

    if issues == 0:
        print("\n🎯 Join verification PASSED: final values exactly match their source dataframes.")
    else:
        print("\n⚠️ Join verification FAILED — see CSVs in", outdir.resolve())

if __name__ == "__main__":
    main()


✅ Driver source unique per DLN (or duplicates have identical values).
✅ Unclaimed source unique per UnclaimedUniqueID (or duplicates have identical values).
✅ All driver keys in final exist in driver source.
✅ All UnclaimedUniqueID values in final exist in unclaimed source.
✅ DriverName equals driver FullName_src: exact match
✅ DriverAddress equals driver FullAddress_src: exact match
✅ UnclaimedName equals unclaimed FullName_cleaned: exact match
✅ UnclaimedAddress equals unclaimed FullAddress_cleaned: exact match
✅ PropertyTypeCodeDescription equals unclaimed source: exact match

=== Join Verification Summary ===
Final rows checked: 6,727,204
Driver key missing in source: 0
Unclaimed key missing in source: 0
DriverName mismatches: 0
DriverAddress mismatches: 0
UnclaimedName mismatches: 0
UnclaimedAddress mismatches: 0
PropertyTypeCodeDescription mismatches: 0

🎯 Join verification PASSED: final values exactly match their source dataframes.


In [16]:
# ===========================================================
# DIRECTOR SUMMARY TABLE — ENTITY RESOLUTION RESULTS
# ===========================================================

import pandas as pd

# --- Load final_df safely ---
try:
    summary_df = final_df.copy()        # if final_df is already in memory
    print("Loaded summary_df from final_df")
except NameError:
    summary_df = pd.read_parquet("final_matches.parquet")
    print("Loaded summary_df from final_matches.parquet")

# ===========================================================
# 1. NORMALIZE VALUE BUCKET
# ===========================================================
vb_norm = (
    summary_df["ValueBucket"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

# One row per property
unique_props = summary_df.assign(ValueBucketNorm=vb_norm).drop_duplicates("UnclaimedUniqueID")

# ===========================================================
# 2. VALUE BUCKET COUNTS
# ===========================================================
count_over    = (unique_props["ValueBucketNorm"] == "OVER $100").sum()
count_under   = (unique_props["ValueBucketNorm"] == "UNDER $100").sum()
count_unknown = (unique_props["ValueBucketNorm"] == "UNKNOWN").sum()

# Minimum estimated value
min_val_over  = count_over  * 100
min_val_under = count_under * 100
min_val_total = min_val_over + min_val_under

# ===========================================================
# 3. MATCH TYPE COUNTS
# ===========================================================
both_count = (summary_df["MatchType"] == "Both").sum()
name_count = (summary_df["MatchType"] == "Name Only").sum()
addr_count = (summary_df["MatchType"] == "Address Only").sum()

# ===========================================================
# 4. COVERAGE — HOW MANY PROPERTIES WE FOUND OWNERS FOR
# ===========================================================
matched_unique_props = unique_props["UnclaimedUniqueID"].nunique()

TOTAL_TREASURY = 1_290_569   # your true original dataset size

unmatched_props = TOTAL_TREASURY - matched_unique_props
match_pct = matched_unique_props / TOTAL_TREASURY * 100

# ===========================================================
# 5. FINAL DIRECTOR SUMMARY TABLE
# ===========================================================
director_summary = pd.DataFrame([
    {"Metric": "Total Treasury Properties (Original Dataset)",
     "Value": f"{TOTAL_TREASURY:,}"},

    {"Metric": "Matched Properties (Any MatchType)",
     "Value": f"{matched_unique_props:,}"},

    {"Metric": "Unmatched Properties (Still 'Lost')",
     "Value": f"{unmatched_props:,}"},

    {"Metric": "% of Properties We Found a Lead For",
     "Value": f"{match_pct:.2f}%"},

    {"Metric": "High-Confidence Matches (Both)",
     "Value": f"{both_count:,}"},

    {"Metric": "Name Only Matches",
     "Value": f"{name_count:,}"},

    {"Metric": "Address Only Matches",
     "Value": f"{addr_count:,}"},

    {"Metric": "Properties Over $100",
     "Value": f"{count_over:,}"},

    {"Metric": "Properties Under $100",
     "Value": f"{count_under:,}"},

    {"Metric": "Properties With Unknown Value",
     "Value": f"{count_unknown:,}"},

    {"Metric": "Minimum $ Returned — Over $100",
     "Value": f"${min_val_over:,.0f}"},

    {"Metric": "Minimum $ Returned — Under $100",
     "Value": f"${min_val_under:,.0f}"},

    {"Metric": "TOTAL Minimum $ Returned (Conservative)",
     "Value": f"${min_val_total:,.0f}"},
])

print("\n===================== DIRECTOR SUMMARY =====================")
print(director_summary.to_string(index=False))


Loaded summary_df from final_df

===================== DIRECTOR SUMMARY =====================
                                      Metric       Value
Total Treasury Properties (Original Dataset)   1,290,569
          Matched Properties (Any MatchType)     470,656
         Unmatched Properties (Still 'Lost')     819,913
         % of Properties We Found a Lead For      36.47%
              High-Confidence Matches (Both)       9,610
                           Name Only Matches     844,664
                        Address Only Matches   5,872,928
                        Properties Over $100     211,991
                       Properties Under $100     183,206
               Properties With Unknown Value      75,459
              Minimum $ Returned — Over $100 $21,199,100
             Minimum $ Returned — Under $100 $18,320,600
     TOTAL Minimum $ Returned (Conservative) $39,519,700


In [17]:
# ===========================================================
# SAVE DIRECTOR SUMMARY TABLE AS CSV
# ===========================================================

DIRECTOR_CSV = "director_summary_table.csv"

director_summary.to_csv(DIRECTOR_CSV, index=False)
print(f"📄 Director summary CSV written to: {DIRECTOR_CSV}")


📄 Director summary CSV written to: director_summary_table.csv


### Unclaimed Property ⇄ Driver License Matching — Simple Brief

#### What we did 

* We tried to find the **right person** for each **unclaimed property** by comparing:

  * **Names** (smart name similarity)
  * **Street addresses** (just the street line, like “123 MAIN ST APT 4”)
* We then labeled each potential match as:

  * **Both** = name **and** address agree
  * **Name Only** = name agrees, address doesn’t
  * **Address Only** = address agrees, name doesn’t
* We cleaned the data (fixed casing, removed junk like “UNKNOWN”, normalized street types) and removed exact duplicates.

---

## 1) Big picture (all potential links we found)

Out of **6,722,031** total potential links (100%):

* **Both (name + address): 9,643** (**0.14%**)
* **Name Only: 839,716** (**12.49%**)
* **Address Only: 5,872,672** (**87.36%**)

**How to read this:** Most potential links come from the address side. The **Both** group is the most trustworthy set to start with.

---

## 2) How much of each source we touched (coverage)

(From our run: about **5.97M drivers** and **1.02M unclaimed** records.)

* **Drivers with at least one potential link:** \~**1,458,500 / 5,967,106** → **24.44%**

* **Unclaimed records with at least one potential link:** \~**469,892 / 1,023,215** → **45.92%**

* **Drivers with a “Both” link (best quality):** \~**2,829 / 5,967,106** → **0.047%**

* **Unclaimed records with a “Both” link:** \~**9,047 / 1,023,215** → **0.884%**

**How to read this:**

* Nearly **half** of unclaimed records have **some** possible match.
* Around **0.9%** of unclaimed records already have **both** signals (our highest confidence group).

---

## 3) One-to-many situations (when the same person or property shows up more than once)

Inside the **Both** group (9,643 rows):

* **Same driver matched to multiple properties:** **782 drivers** (7,596 rows)
  →  (one person can have several unclaimed items).
* **Same property matched to multiple drivers:** **592 properties** (1,188 rows)
  → **Needs review** (we should pick the single best person for that property).

We have **no duplicate pairs** left after cleanup (no repeated “same driver + same property” rows).

---

## 4) About “exact” and “strong” matches

* We track when scores are **exact 1.00** (perfect) for name or address.
* Our “strong” cutoffs are **Name ≥ 0.90** and **Address ≥ 0.95**.
* Tiny score quirks above 1.00 can happen due to math rounding; we clip them when exporting.

---

## 5) Why there are many “Address Only” links

1. Many people can share the **same address** (apartments, PO Boxes).
2. We used a **strict bar for names** to avoid false matches.
3. Our **blocking** narrows comparisons but still lets address-only links dominate in count.

**Action:** Start with the **Both** group for outreach. For “Address Only”, add extra checks (valid ZIP, not a placeholder, DOB or email if available).

---

## 6) Data quality protections

* Cleaned names/addresses (removed junk, normalized suffixes like JR/SR, street types like ST/AVE, handled ZIP “00000” as missing).
* Dropped obvious placeholder addresses.
* Kept **one row per driver–property pair** with the **best scores**.
* Produced quality reports to spot:

  * drivers with multiple properties,
  * properties with multiple drivers (review needed).

---

## 7) What to do next

1. **Work the “Both” list first**: 9,643 high-confidence rows.
2. **Resolve properties with multiple drivers**: pick the highest **NameScore**, then **AddressScore** as a tiebreaker; use DOB/age if available.
3. **Enrich “Address Only”** before outreach (DOB year, phone/email, prior address links).
4. **Sample-check** each bucket (Both / Name Only / Address Only) to confirm precision before large-scale contact.

---

### Quick takeaways for leadership

* We created a reliable pipeline at state scale (millions of records).
* **9,643 “Both” matches** are our **best** immediate candidates for returning property.
* Almost **half** of unclaimed records now have **some** lead; a focused review will turn many into confirmed returns.
* We have clear rules to resolve conflicts and avoid contacting the wrong person.


### Bigquery

In [ ]:
from google.cloud import bigquery

# --- EDIT THESE ---
BOTH_CSV    = "final_matches_both.csv"       # or "gs://bucket/final_matches_both.csv"
JOINED_CSV  = "final_matches_joined.csv"     # or "gs://bucket/final_matches_joined.csv"
BOTH_TABLE  = "omes-datascience-sbx.Unclaimed_Property_Analysis.final_matches_both"
JOINED_TABLE= "omes-datascience-sbx.Unclaimed_Property_Analysis.final_matches_joined"
# ------------------

client = bigquery.Client()
job_cfg = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    autodetect=True,
    skip_leading_rows=1,           # header row
    field_delimiter=",",
    quote_character='"',
    allow_quoted_newlines=True,
    write_disposition="WRITE_TRUNCATE",
    create_disposition="CREATE_IF_NEEDED",
)

def load_csv_to_bq(source, table):
    if source.startswith("gs://"):
        job = client.load_table_from_uri(source, table, job_config=job_cfg, location="us-central1")
    else:
        with open(source, "rb") as f:
            job = client.load_table_from_file(f, table, job_config=job_cfg, location="us-central1")
    job.result()
    print(f"Loaded {client.get_table(table).num_rows:,} rows → {table}")

load_csv_to_bq(BOTH_CSV,   BOTH_TABLE)
load_csv_to_bq(JOINED_CSV, JOINED_TABLE)
